# ICS 591 Team 2 — Stage 2b: Model-assisted labelling of the local dataset

Builds on Stage 2a, using the images and `candidates.csv` already in Drive, and on the Stage 1 ViT-S/16 checkpoint.

## The plan
1. **Human test set (150 images).** All three of us label the same images **from scratch, with no hint**. The majority vote is the label. These images are never auto-labelled or trained on; they are what we report final results on. They also give us inter-annotator agreement (Cohen's kappa).
2. **Seed set (about 300 images).** CLIP's suggestion is pre-selected; you confirm with one click or correct it. Split across the three of us.
3. **Round *k*:**
   - Fine-tune the Stage 1 ViT on all human labels plus earlier auto-labels.
   - Measure how accurate it is **at each confidence level**, using cross-validation on the human labels.
   - Pick a confidence threshold per class where that accuracy is at least 95%.
   - **Auto-label** pool images above the threshold.
   - Send the **most uncertain** images to humans, which is active learning.
   - Send a **random audit sample** of the auto-labels to humans, with no hint, to measure their real accuracy.
4. **Repeat** for 2–3 rounds, until the audit accuracy is at least about 95% and few uncertain images remain.
5. **Export** `dataset_local/train` and `dataset_local/test`, with a manifest recording where each label came from (human, model or audit).

The model learns 4 classes: `plastic_clean`, `plastic_contaminated`, `non_plastic` and `discard`. The `discard` class lets the model recognise and throw out junk images, such as posters, logos and photos with no waste, instead of forcing them into a waste class.

## Who runs what
| Cell | Who | When |
|---|---|---|
| 1, 2 | everyone | every session |
| 3 *Plan pools* | **one person, once** | at the start |
| 4 *Review tool* | **everyone**, with their own name | whenever there is work in the queue |
| 5 *Progress & agreement* | anyone | any time |
| 6 *Train round* + 7 *Auto-label* | **one person** (needs a GPU) | when the queue is empty, then set `ROUND = ROUND + 1` |
| 8 *Audit results* | anyone | after each round's review |
| 9 *Export* | one person | at the end |

Use **Runtime → T4 GPU** for cells 6–7. The review tool works without a GPU. Don't use *Run all*.

**Labelling rules** are the same as in Stage 2a: label what dominates the image; when unsure between clean and contaminated, choose contaminated; discard identifiable people.

In [ ]:
# 1. Setup (everyone, every session)
!pip -q install timm
import os, io, json, time, math, random, shutil, pathlib
import numpy as np, pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from google.colab import drive
drive.mount('/content/drive')

OUT        = pathlib.Path('/content/drive/MyDrive/ics591_local_data')
IMG_DRIVE  = OUT / 'raw' / 'images'
STAGE1_CKPT = pathlib.Path('/content/drive/MyDrive/ics591_model_selection/realwaste/ViT-S_16.pt')
TIMM_ID    = 'vit_small_patch16_224.augreg_in21k_ft_in1k'
assert (OUT / 'candidates.csv').exists(), 'candidates.csv not found - run Stage 2a cells 1-6 first'
print('Stage 1 checkpoint found:', STAGE1_CKPT.exists())

In [ ]:
# 2. Shared settings and helpers (everyone, every session)
TEAM = ['nana', 'kelvin', 'cajetan']
CLASSES = ['plastic_clean', 'plastic_contaminated', 'non_plastic']
MODEL_CLASSES = CLASSES + ['discard']
BUTTONS = CLASSES + ['discard_junk', 'discard_person']

N_TEST, N_SEED = 150, 300
TARGET_PRECISION = 0.95   # auto-label only where the model is measured to be >= 95% right
MIN_SUPPORT = 10          # need at least this many held-out human examples to trust a threshold
REVIEW_BUDGET = 300       # uncertain images sent to humans per round (split across the team)
AUDIT_N = 45              # random auto-labels re-checked by humans per round (15 each)

QUEUE  = OUT / 'al_queue.csv'          # what each person has to label
AUTO   = OUT / 'al_auto_labels.csv'    # labels assigned by the model
ROUNDS = OUT / 'al_rounds.json'        # metrics per round
def REV(p): return OUT / f'al_reviews_{p}.csv'   # each person's labels

def to_model_label(l):
    return 'discard' if str(l).startswith('discard') else l

def read_csv_or_empty(path, cols):
    return pd.read_csv(path) if pathlib.Path(path).exists() else pd.DataFrame(columns=cols)

def load_reviews():
    parts = [pd.read_csv(REV(p)) for p in TEAM if REV(p).exists()]
    if not parts: return pd.DataFrame(columns=['file', 'label', 'pool', 'labeler', 'ts'])
    return pd.concat(parts, ignore_index=True)

def test_labels():
    # majority vote over the test pool; ties are dropped as too ambiguous to grade on
    r = load_reviews(); r = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
    out = []
    for f, g in r.groupby('file'):
        vc = g['label'].map(to_model_label).value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]:
            out.append({'file': f, 'label': vc.index[0], 'n_votes': len(g)})
    return pd.DataFrame(out, columns=['file', 'label', 'n_votes'])

def test_files():
    q = read_csv_or_empty(QUEUE, ['file', 'pool'])
    return set(q.loc[q['pool'] == 'test', 'file'])

def current_labels():
    # Human labels (non-test) always override model labels.
    r = load_reviews(); r = r[r['pool'] != 'test'].sort_values('ts').drop_duplicates('file', keep='last')
    human = pd.DataFrame({'file': r['file'], 'label': r['label'].map(to_model_label), 'raw_label': r['label'],
                          'source': np.where(r['pool'].astype(str).str.startswith('audit'), 'human_audit', 'human')})
    auto = read_csv_or_empty(AUTO, ['file', 'label', 'round', 'conf'])
    auto = auto[~auto['file'].isin(human['file'])]
    auto = pd.DataFrame({'file': auto['file'], 'label': auto['label'], 'raw_label': auto['label'], 'source': 'model'})
    lab = pd.concat([human, auto], ignore_index=True)
    return lab[~lab['file'].isin(test_files())]

def pick_thresholds(probs, y, target=TARGET_PRECISION, min_support=MIN_SUPPORT):
    # For each predicted class: the lowest confidence t such that, among held-out human-labelled images
    # predicted as that class with confidence >= t, at least `target` are correct (and >= min_support images).
    conf, pred = probs.max(1), probs.argmax(1); thr = {}
    for c, name in enumerate(MODEL_CLASSES):
        m = pred == c; thr[name] = float('inf')
        for t in np.sort(np.unique(conf[m])):
            sel = m & (conf >= t)
            if sel.sum() < min_support: break
            if (y[sel] == c).mean() >= target:
                thr[name] = float(t); break
    return thr

def assign_round_robin(n, start=0):
    return [TEAM[(start + i) % len(TEAM)] for i in range(n)]

In [ ]:
# 3. PLAN POOLS — ONE person runs this ONCE at the very start
if QUEUE.exists():
    print('Queue already exists - pools were planned before. Nothing to do.')
else:
    cand = pd.read_csv(OUT / 'candidates.csv')
    if 'suggestion' not in cand: cand['suggestion'], cand['suggestion_conf'] = np.nan, np.nan

    # Import any labels already made with the Stage 2a tool, so that work isn't lost
    prior = []
    for p in TEAM:
        f = OUT / f'labels_{p}.csv'
        if f.exists():
            d = pd.read_csv(f).sort_values('ts').drop_duplicates('file', keep='last')
            d = d[d['label'].isin(BUTTONS)]
            d['pool'], d['labeler'] = 'seed_stage2a', p
            prior.append(d[['file', 'label', 'pool', 'labeler', 'ts']])
    prior = pd.concat(prior, ignore_index=True).drop_duplicates('file') if prior else pd.DataFrame()
    for p in TEAM:
        part = prior[prior['labeler'] == p] if len(prior) else prior
        if len(part): part.to_csv(REV(p), index=False)
    print(f'Imported {len(prior)} labels from Stage 2a')

    prior_files = set(prior['file']) if len(prior) else set()
    rest = cand[~cand['file'].isin(prior_files)].sample(frac=1, random_state=42).reset_index(drop=True)
    n_seed = max(0, N_SEED - len(prior_files))
    test, seed = rest.iloc[:N_TEST], rest.iloc[N_TEST:N_TEST + n_seed]

    q = pd.concat([
        pd.DataFrame({'file': test['file'], 'pool': 'test', 'assigned': 'ALL', 'hint': '', 'hint_conf': np.nan}),
        pd.DataFrame({'file': seed['file'], 'pool': 'seed', 'assigned': assign_round_robin(len(seed)),
                      'hint': seed['suggestion'].fillna(''), 'hint_conf': seed['suggestion_conf']}),
    ], ignore_index=True)
    q['order'] = range(len(q))
    q.to_csv(QUEUE, index=False)
    print(f'Test pool: {len(test)} (labelled by all 3)  |  seed pool: {len(seed)}  |  '
          f'unlabelled pool left: {len(rest) - len(test) - len(seed)}')
    print(q.groupby(['pool', 'assigned']).size())

In [ ]:
# 4. REVIEW TOOL — everyone, with their own name. Works through everything assigned to you.
LABELER = 'nana'      # <-- change to your name: 'nana', 'kelvin' or 'cajetan'

import ipywidgets as W
from IPython.display import display
assert LABELER in TEAM
q = pd.read_csv(QUEUE)
mine = q[q['assigned'].isin(['ALL', LABELER])].sort_values('order')
mine_rev = read_csv_or_empty(REV(LABELER), ['file', 'label', 'pool', 'labeler', 'ts'])
done = dict(zip(mine_rev['file'], mine_rev['label']))
items = mine.to_dict('records')
pos = next((i for i, r in enumerate(items) if r['file'] not in done), len(items))

img, meta, prog = W.Image(format='jpeg', width=560), W.HTML(), W.HTML()
NAMES = {'plastic_clean': 'plastic clean', 'plastic_contaminated': 'plastic contaminated',
         'non_plastic': 'non-plastic', 'discard_junk': 'discard: junk / no waste',
         'discard_person': 'discard: identifiable person'}
btn = {k: W.Button(description=v, layout=W.Layout(width='220px')) for k, v in NAMES.items()}
back = W.Button(description='◀ back', layout=W.Layout(width='100px'))

def show():
    n_done = sum(r['file'] in done for r in items)
    prog.value = f'<b>{LABELER}</b>: {n_done}/{len(items)} done'
    for k, b in btn.items(): b.button_style = ''; b.description = NAMES[k]
    if pos >= len(items):
        img.value = b''; meta.value = '<h3>Your queue is empty — thank you!</h3>'; return
    r = items[pos]; img.value = (IMG_DRIVE / r['file']).read_bytes()
    hint = r['hint'] if isinstance(r['hint'], str) else ''
    blind = r['pool'] == 'test' or str(r['pool']).startswith('audit')
    if blind or not hint:
        note = '<b>Label from scratch</b> (no hint shown)'
    else:
        note = f'Suggested: <b>{NAMES.get(hint, hint)}</b> — click it to confirm, or pick another'
        if hint in btn: btn[hint].button_style = 'success'; btn[hint].description = '✓ ' + NAMES[hint]
    meta.value = f"#{pos + 1} · pool: {r['pool']} · {note} · current: <b>{done.get(r['file'], '—')}</b>"

def on_label(k):
    def h(_):
        global pos
        if pos >= len(items): return
        r = items[pos]; done[r['file']] = k
        pd.DataFrame([{'file': r['file'], 'label': k, 'pool': r['pool'], 'labeler': LABELER, 'ts': time.time()}]) \
          .to_csv(REV(LABELER), mode='a', header=not REV(LABELER).exists(), index=False)
        pos += 1; show()
    return h

def on_back(_):
    global pos
    pos = max(0, pos - 1); show()

for k, b in btn.items(): b.on_click(on_label(k))
back.on_click(on_back)
display(W.VBox([prog, img, meta, W.HBox([btn[c] for c in CLASSES]),
                W.HBox([btn['discard_junk'], btn['discard_person'], back])]))
show()

In [ ]:
# 5. PROGRESS AND AGREEMENT (anyone, any time)
from sklearn.metrics import cohen_kappa_score
from itertools import combinations
q, r = pd.read_csv(QUEUE), load_reviews()
print('Outstanding work per person:')
for p in TEAM:
    mine = q[q['assigned'].isin(['ALL', p])]
    did = set(r.loc[r['labeler'] == p, 'file'])
    print(f'  {p:8s} {len(mine) - mine["file"].isin(did).sum():4d} left of {len(mine)}')

t = r[r['pool'] == 'test'].sort_values('ts').drop_duplicates(['file', 'labeler'], keep='last')
w = t.pivot(index='file', columns='labeler', values='label')
print('\nTest-set agreement (Cohen kappa; > 0.6 = substantial):')
for a, b in combinations([c for c in TEAM if c in w], 2):
    both = w[[a, b]].dropna()
    if len(both):
        print(f'  {a} vs {b}: n={len(both)}, raw agreement {(both[a] == both[b]).mean():.1%}, '
              f'kappa {cohen_kappa_score(both[a], both[b]):.2f}')
tl = test_labels()
print(f'\nTest images with a majority label: {len(tl)}'); print(tl['label'].value_counts())
cl = current_labels()
print('\nTraining labels so far:'); print(pd.crosstab(cl['label'], cl['source'], margins=True))

In [ ]:
# 6. TRAIN A ROUND — one person, GPU. Run when everyone's queue is empty.
ROUND = 1             # <-- 1, then 2, then 3 ...
FOLDS = 3             # cross-validation folds used to measure accuracy at each confidence level
EPOCHS = 10
BATCH = 32

import torch, timm, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from concurrent.futures import ThreadPoolExecutor
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEV.type == 'cuda', 'Switch to a GPU runtime for this cell'

LOCAL = pathlib.Path('/content/img_cache'); LOCAL.mkdir(exist_ok=True)
def cache(files):  # copy images from Drive to local disk once (much faster training)
    todo = [f for f in files if not (LOCAL / f).exists()]
    with ThreadPoolExecutor(16) as ex: list(ex.map(lambda f: shutil.copy(IMG_DRIVE / f, LOCAL / f), todo))

def build_model():
    if STAGE1_CKPT.exists():
        m = timm.create_model(TIMM_ID, pretrained=False, num_classes=9)
        m.load_state_dict(torch.load(STAGE1_CKPT, map_location='cpu'))
        m.reset_classifier(len(MODEL_CLASSES))
    else:
        print('Stage 1 checkpoint not found - starting from ImageNet weights')
        m = timm.create_model(TIMM_ID, pretrained=True, num_classes=len(MODEL_CLASSES))
    return m.to(DEV)

_cfg = timm.data.resolve_data_config({}, model=timm.create_model(TIMM_ID, pretrained=False))
NORM = T.Normalize(_cfg['mean'], _cfg['std'])
TRAIN_TF = T.Compose([T.RandomResizedCrop(224, scale=(0.5, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3), T.ToTensor(), NORM])
EVAL_TF = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), NORM])

class DS(Dataset):
    def __init__(self, files, ys, tf): self.f, self.y, self.tf = list(files), list(ys), tf
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.tf(Image.open(LOCAL / self.f[i]).convert('RGB')), self.y[i]

def train(files, ys, seed=0):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    m = build_model(); ys = np.asarray(ys)
    dl = DataLoader(DS(files, ys, TRAIN_TF), BATCH, shuffle=True, num_workers=2, drop_last=len(ys) > BATCH)
    head = list(m.get_classifier().parameters()); hid = {id(p) for p in head}
    body = [p for p in m.parameters() if id(p) not in hid]
    opt = torch.optim.AdamW([{'params': body, 'lr': 3e-5}, {'params': head, 'lr': 1e-3}], weight_decay=0.05)
    steps = EPOCHS * len(dl); sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[3e-5, 1e-3], total_steps=steps, pct_start=0.1)
    counts = np.bincount(ys, minlength=len(MODEL_CLASSES)).astype(float)
    w = torch.tensor(np.where(counts > 0, counts.sum() / np.maximum(counts, 1) / len(MODEL_CLASSES), 0), dtype=torch.float32, device=DEV)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda')
    m.train()
    for ep in range(EPOCHS):
        for x, y in dl:
            x, y = x.to(DEV), y.to(DEV); opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda', dtype=torch.float16): loss = crit(m(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return m.eval()

@torch.no_grad()
def predict(m, files):
    dl = DataLoader(DS(files, [0] * len(files), EVAL_TF), 64, num_workers=2); P = []
    for x, _ in dl:
        with torch.autocast('cuda', dtype=torch.float16): P.append(m(x.to(DEV)).float().softmax(1).cpu())
    return torch.cat(P).numpy() if P else np.zeros((0, len(MODEL_CLASSES)))

from scipy.optimize import minimize_scalar
def fit_T(p, y):
    lp = np.log(np.clip(p, 1e-12, 1))
    def nll(t):
        z = lp / t; z -= z.max(1, keepdims=True); z -= np.log(np.exp(z).sum(1, keepdims=True))
        return -z[np.arange(len(y)), y].mean()
    return float(minimize_scalar(nll, bounds=(0.05, 10), method='bounded').x)
def apply_T(p, t):
    z = np.log(np.clip(p, 1e-12, 1)) / t; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

# ---- data for this round
lab = current_labels(); lab['y'] = lab['label'].map(MODEL_CLASSES.index)
human = lab[lab['source'] != 'model'].reset_index(drop=True); auto = lab[lab['source'] == 'model']
tl = test_labels(); tl['y'] = tl['label'].map(MODEL_CLASSES.index)
print(f'Round {ROUND}: {len(human)} human labels, {len(auto)} model labels, {len(tl)} test images')
print(pd.crosstab(lab['label'], lab['source']))
if (human['y'].value_counts().reindex(range(len(MODEL_CLASSES)), fill_value=0) < FOLDS).any():
    print('WARNING: a class has very few human labels - thresholds for it will stay off (no auto-labels).')
cache(list(lab['file']) + list(tl['file']))

# ---- cross-validated predictions on human labels (to measure accuracy at each confidence level)
oof = np.zeros((len(human), len(MODEL_CLASSES)))
try: splits = list(StratifiedKFold(FOLDS, shuffle=True, random_state=ROUND).split(human, human['y']))
except ValueError: splits = list(KFold(FOLDS, shuffle=True, random_state=ROUND).split(human))
for k, (tr, va) in enumerate(splits):
    t0 = time.time()
    f_tr = list(human.loc[tr, 'file']) + list(auto['file']); y_tr = list(human.loc[tr, 'y']) + list(auto['y'])
    m = train(f_tr, y_tr, seed=k); oof[va] = predict(m, list(human.loc[va, 'file']))
    print(f'  fold {k + 1}/{FOLDS}: {time.time() - t0:.0f}s, fold accuracy {(oof[va].argmax(1) == human.loc[va, "y"]).mean():.3f}')
    del m; torch.cuda.empty_cache()

TEMP = fit_T(oof, human['y'].values); oof_cal = apply_T(oof, TEMP)
THR = pick_thresholds(oof_cal, human['y'].values)

# ---- final model for this round, on everything labelled
t0 = time.time(); MODEL = train(list(lab['file']), list(lab['y']), seed=100 + ROUND)
torch.save(MODEL.state_dict(), OUT / f'al_model_round{ROUND}.pt'); print(f'final model trained in {time.time() - t0:.0f}s')

# ---- report on the human test set (never used for any decision)
pt = apply_T(predict(MODEL, list(tl['file'])), TEMP); yt = tl['y'].values
res = {'round': ROUND, 'n_human': int(len(human)), 'n_model_labels': int(len(auto)), 'temperature': TEMP,
       'thresholds': THR, 'cv_accuracy': float((oof.argmax(1) == human['y']).mean()),
       'test_n': int(len(yt)), 'test_accuracy': float(accuracy_score(yt, pt.argmax(1))),
       'test_macro_f1': float(f1_score(yt, pt.argmax(1), average='macro', labels=range(len(MODEL_CLASSES))))}
hist = json.loads(ROUNDS.read_text()) if ROUNDS.exists() else []
hist = [h for h in hist if h['round'] != ROUND] + [res]; ROUNDS.write_text(json.dumps(hist, indent=2))
print(json.dumps(res, indent=2))
print(pd.DataFrame(confusion_matrix(yt, pt.argmax(1), labels=range(len(MODEL_CLASSES))),
                   index=['true ' + c for c in MODEL_CLASSES], columns=['pred ' + c for c in MODEL_CLASSES]))

In [ ]:
# 7. AUTO-LABEL THE POOL AND QUEUE WORK FOR HUMANS (same person, right after cell 6)
cand = pd.read_csv(OUT / 'candidates.csv'); q = pd.read_csv(QUEUE)
reviewed = set(load_reviews()['file'])
pending = set(q['file']) - reviewed                 # queued but not yet labelled
taken = set(current_labels()['file']) | test_files() | pending
pool = [f for f in cand['file'] if f not in taken]
print(f'Unlabelled pool: {len(pool)} images')
cache(pool)
pp = apply_T(predict(MODEL, pool), TEMP)
conf, pred = pp.max(1), pp.argmax(1)
srt = np.sort(pp, 1); margin = srt[:, -1] - srt[:, -2]
thr_arr = np.array([THR[c] for c in MODEL_CLASSES])
accept = conf >= thr_arr[pred]

new_auto = pd.DataFrame({'file': np.array(pool)[accept], 'label': np.array(MODEL_CLASSES)[pred[accept]],
                         'round': ROUND, 'conf': conf[accept].round(4)})
new_auto.to_csv(AUTO, mode='a', header=not AUTO.exists(), index=False)

unc_idx = np.where(~accept)[0]; unc_idx = unc_idx[np.argsort(margin[unc_idx])][:REVIEW_BUDGET]
hint = lambda c: 'discard_junk' if c == 'discard' else c
rev = pd.DataFrame({'file': np.array(pool)[unc_idx], 'pool': f'round{ROUND}',
                    'assigned': assign_round_robin(len(unc_idx)),
                    'hint': [hint(MODEL_CLASSES[c]) for c in pred[unc_idx]], 'hint_conf': conf[unc_idx].round(3)})
aud = new_auto.sample(min(AUDIT_N, len(new_auto)), random_state=ROUND)
aud = pd.DataFrame({'file': aud['file'], 'pool': f'audit{ROUND}', 'assigned': assign_round_robin(len(aud), 1),
                    'hint': '', 'hint_conf': np.nan})
add = pd.concat([rev, aud], ignore_index=True); add['order'] = q['order'].max() + 1 + np.arange(len(add))
pd.concat([q, add], ignore_index=True).to_csv(QUEUE, index=False)

print(f'Auto-labelled this round: {accept.sum()}'); print(new_auto['label'].value_counts())
print(f'\nSent to humans: {len(rev)} uncertain + {len(aud)} audit  |  still unlabelled after this: {(~accept).sum() - len(rev)}')
print('Thresholds used:', {k: (round(v, 3) if np.isfinite(v) else 'off') for k, v in THR.items()})
print('\nNext: everyone runs the review tool (cell 4); then check cell 8; then rerun cells 6-7 with ROUND + 1.')

In [ ]:
# 8. AUDIT RESULTS — how accurate were the model's auto-labels? (run after the round's review is done)
auto = read_csv_or_empty(AUTO, ['file', 'label', 'round', 'conf'])
r = load_reviews(); a = r[r['pool'].astype(str).str.startswith('audit')].sort_values('ts').drop_duplicates('file', keep='last')
m = a.merge(auto, on='file', suffixes=('_human', '_model'))
m['correct'] = m['label_human'].map(to_model_label) == m['label_model']
summ = m.groupby('round').agg(audited=('correct', 'size'), accuracy=('correct', 'mean'))
print(summ)
hist = json.loads(ROUNDS.read_text()) if ROUNDS.exists() else []
print(pd.DataFrame(hist)[['round', 'n_human', 'n_model_labels', 'cv_accuracy', 'test_accuracy', 'test_macro_f1']]
      if hist else 'no rounds yet')
print('\nStop when audit accuracy >= ~95% and few uncertain images remain (usually 2-3 rounds).')
print('If audit accuracy is low, raise TARGET_PRECISION in cell 2 (e.g. 0.97) before the next round.')

In [ ]:
# 9. EXPORT THE FINAL DATASET (one person, at the end)
DATA = OUT / 'dataset_local'
lab = current_labels(); tl = test_labels()
q = pd.read_csv(QUEUE); r = load_reviews()
train_set = lab[lab['label'].isin(CLASSES)]; test_set = tl[tl['label'].isin(CLASSES)]
if DATA.exists(): shutil.rmtree(DATA)
for split, df in [('train', train_set), ('test', test_set)]:
    for c in CLASSES: (DATA / split / c).mkdir(parents=True, exist_ok=True)
    for f, c in zip(df['file'], df['label']): shutil.copy(IMG_DRIVE / f, DATA / split / c / f)

cand = pd.read_csv(OUT / 'candidates.csv')[['file', 'query', 'image_url', 'source_page']]
man = pd.concat([train_set.assign(split='train')[['file', 'label', 'source', 'split']],
                 test_set.assign(split='test', source='human_majority')[['file', 'label', 'source', 'split']]])
man.merge(cand, on='file', how='left').to_csv(OUT / 'dataset_local_manifest.csv', index=False)

print('TRAIN (label source x class):'); print(pd.crosstab(train_set['label'], train_set['source'], margins=True))
print('\nTEST (human majority vote):'); print(test_set['label'].value_counts())
print(f"\nDiscarded as junk/person: {(lab['label'] == 'discard').sum()} train-pool + {(tl['label'] == 'discard').sum()} test-pool")
print(f"Never labelled (left uncertain): {len(set(cand['file']) - set(lab['file']) - set(tl['file']) - test_files())}")
print('\nWritten to', DATA, '- this is the input to the Stage 2c fine-tuning notebook.')